# Recyclage et territoire : à la recherche d'un profil écologique géographique ?

## Problématique

Les questions écologiques constituent une thématique importante notamment dans la gestion des territoires. Partant de ce constat, nous avons cherché un indicateur de comportement écologique que l'on pourrait croiser avec des données géographiques liés à la population francaise. Nous avons donc décidé de nous interesser à la **valorisation des déchets ménagers et assimilés (DMA)**. 

Nous allons montrer que cette valorisation n'est pas la même au sein de la France métropolitaine. En effet, les performances de valorisation varient considérablement d'un département à l'autre. Quels facteurs territoriaux et socio-économiques expliquent ces disparités ?

Ce notebook explore deux hypothèses principales :
1. **Hypothèse territoriale** : les départements ruraux, bénéficiant d'une gestion différente des déchets (compostage, valorisation organique), présenteraient de meilleurs taux de valorisation.
2. **Hypothèse socio-économique** : les départements plus aisés investiraient davantage dans les infrastructures de tri et de recyclage.

Nous nous sommes aussi intéressés aux deux composantes de la valorisation :
- ** Valorisation de la matière ** : recyclage
- ** Valorisation organique ** : compostage et méthanisation

## Données utilisées

| Source | Fichier | Description |
|--------|---------|-------------|
| ADEME / SINOE | `SINOE04_DestinationDmaParTypeTraitement.csv` | Tonnages de déchets par type de traitement et département |
| INSEE | `FET2021-19.xlsx` | Grille de densité communale (urbain/rural) |
| INSEE | `niv2021.xlsx` | Niveau de vie médian annuel par département(légèrement revu, les 3 premiers lignes ont été supprimées |

> **Reproductibilité** : les fichiers de données doivent être placés dans le même répertoire que ce notebook. Les sources sont disponibles sur [data.ademe.fr](https://data.ademe.fr) et [insee.fr](https://insee.fr).

## Plan
0. Packages et impots
1. Chargement et nettoyage des données
2. Analyse descriptive
3. Visualisations : distributions, nuages de points, cartes
4. Modélisation : corrélations et régression multiple
5. Conclusion

## 0. Quelques package necessaires : 

Installer les packages 

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr, spearmanr

# ── Paramètres globaux ─────────────────────────────────────────────────────────
# dpi=120 : résolution des figures (120 points par pouce, plus net que le défaut)
plt.rcParams["figure.dpi"] = 120
# whitegrid : fond blanc avec grille légère ; palette "muted" : couleurs douces
sns.set_theme(style="whitegrid", palette="muted")

# URL du fond de carte GeoJSON 
# contours des 96 départements métropolitains
# 
GEOJSON_URL = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"

print("Imports OK")

## 1. Chargement et nettoyage des données

### 1.1 Données SINOE (déchets)
 Le fichier SINOE recense les tonnages de DMA par département, par année
 et par **type de traitement**. Les types disponibles sont :
 - Valorisation matière ← recyclage
 - Valorisation organique ← compostage / méthanisation
 - Incinération avec/sans récupération d'énergie
 - Stockage
 - autre


In [ ]:
def charger_sinoe(path: str) -> pd.DataFrame:
    """Charge et nettoie les données SINOE.
    
    Retourne un DataFrame avec une ligne par département et les colonnes :
    - tonnage_total, tonnage_valo_matiere, tonnage_valo_organique
    - taux_valo_total_pct, taux_valo_matiere_pct, taux_valo_organique_pct
    """
    # lecture du fichier csv


    df = pd.read_csv(path, sep=",", encoding="utf-8")
    print(f"Dimensions brutes : {df.shape}")
    print(f"Années disponibles : {sorted(df['ANNEE'].unique())}")
    print(f"Types de traitement : {df['L_TYP_REG_SERVICE'].unique().tolist()}")

    # Nettoyage du tonnage (virgule devient point pour python). On passe ensuite en float
    df["TONNAGE_DMA"] = (
        df["TONNAGE_DMA"]
        .astype(str)                          
        .str.replace(",", ".", regex=False)   
        .astype(float)                       
    )

    # Les codes département doivent être sur 2 caractères pour les jointures
    df["code_dept"]   = df["C_DEPT"].astype(str).str.zfill(2)
    df["departement"] = df["N_DEPT"]

    # Conserver la dernière année disponible
    annee_max = df["ANNEE"].max()
    df        = df[df["ANNEE"] == annee_max].copy()
    print(f"\nAnnée retenue : {annee_max}  ({len(df)} lignes après filtre)")

    # On crée des colonnes de tonnage conditionnelles AVANT le groupby.
    # where(condition, 0) : garde le tonnage si la condition est vraie, sinon met 0.
    df["est_valo_matiere"]   = df["L_TYP_REG_SERVICE"] == "Valorisation matière"
    df["est_valo_organique"] = df["L_TYP_REG_SERVICE"] == "Valorisation organique"
    df["tonnage_si_matiere"]   = df["TONNAGE_DMA"].where(df["est_valo_matiere"],   0)
    df["tonnage_si_organique"] = df["TONNAGE_DMA"].where(df["est_valo_organique"], 0)

    # Pour chaque département, on somme les tonnages selon leur type.
    
    dept = (
        df.groupby(["code_dept", "departement"], as_index=False)
        .agg(
            tonnage_total          = ("TONNAGE_DMA",          "sum"),
            tonnage_valo_matiere   = ("tonnage_si_matiere",   "sum"),
            tonnage_valo_organique = ("tonnage_si_organique", "sum"),
        )
    )

    # ── Calcul des taux de valorisation (en %) ───────────────────────────────────
    # Taux = tonnage valorisé / tonnage total × 100
    dept["tonnage_valo_total"]      = dept["tonnage_valo_matiere"] + dept["tonnage_valo_organique"]
    dept["taux_valo_total_pct"]     = dept["tonnage_valo_total"]    / dept["tonnage_total"] * 100
    dept["taux_valo_matiere_pct"]   = dept["tonnage_valo_matiere"]  / dept["tonnage_total"] * 100
    dept["taux_valo_organique_pct"] = dept["tonnage_valo_organique"]/ dept["tonnage_total"] * 100

    print(f"{len(dept)} départements après agrégation")
    return dept
    

In [ ]:
# Appel de la fonction charger_sinoe
sinoe_dept = charger_sinoe("SINOE04_DestinationDmaParTypeTraitement.csv")
display(sinoe_dept.head())

### 1.2 Données FET (ruralité)

La grille de densité classe chaque commune en 4 catégories:
Commnune densément peuplée, densité intermédiaire, peu dense et très peu dense. On considère les deux dernières comme rurale.

On utilise comme indicateur la part de communes rurales dans le département.




In [ ]:
def charger_ruralite(path: str) -> pd.DataFrame:
    """Charge et nettoie les données FET (grille de densité INSEE).
    
    Retourne un DataFrame à l'échelle départementale avec :
    - part_communes_rurales_pct : % de communes peu ou très peu denses
    
    Remarque : l'indicateur n'est pas pondéré par la population,
    ce qui peut surreprésenter des communes faiblement peuplées.
    """
    df = pd.read_excel(path, sheet_name="Figure 1", skiprows=2)
    # skiprows=2 : les deux premières lignes sont un en-tête inutile

    df.columns = ["code_commune", "lib_commune", "code_typologie", "lib_typologie"]

    print(f"{len(df)} communes chargées")
    print(f"Typologies : {df['lib_typologie'].unique().tolist()}")

    # Codes communes sur 5 caractères  
    df["code_commune"] = df["code_commune"].astype(str).str.zfill(5)

    # Codes département sur les 2 premiers caractères 
    df["code_dept"] = df["code_commune"].str[:2]

    # On identifie les communes rurales
    types_ruraux = ["Communes peu denses", "Communes très peu denses"]
    df["est_rural"] = df["lib_typologie"].isin(types_ruraux)

   # On agrège par département
    dept = (
        df.groupby("code_dept", as_index=False)
        .agg(
            nb_communes=("code_commune", "count"),
            nb_communes_rurales=("est_rural", "sum"),
        )
    )
    # On calcule la part des communes rurales
    dept["part_communes_rurales_pct"] = (
        dept["nb_communes_rurales"] / dept["nb_communes"] * 100
    )
    print(f"{len(dept)} départements après agrégation")
    return dept


In [ ]:
# On appelle la fonction
    
ruralite_dept = charger_ruralite("FET2021-19.xlsx")
display(ruralite_dept.head())

### 1.3 Données INSEE (niveau de vie médian)

In [ ]:
def charger_niveau_vie(path: str) -> pd.DataFrame:
    """Charge et nettoie les données INSEE sur le niveau de vie médian.
    
    Retourne un DataFrame à l'échelle départementale avec :
    - niveau_vie_median : niveau de vie annuel médian en euros
    """

    # Lecture
    df = pd.read_excel(path, sheet_name="Territoire - Figure 1")
    print(f"Colonnes disponibles : {df.columns.tolist()}")
    print(f"Dimensions : {df.shape}")

    # quelques changements de nom, code_dept pour harmoniser et departement_rev 
    # pour département revenu
    df = df.rename(columns={
        "Code département": "code_dept",
        "Département": "departement_rev",
        "Niveau de vie annuel médian": "niveau_vie_median",
    })
    
    # Codes département sur les 2 premiers caractères 
    df["code_dept"] = df["code_dept"].astype(str).str.zfill(2)

    #  Conversion numérique 
    #  si une valeur n'est pas un nombre alors on l'enregistre comme NaN 
    df["niveau_vie_median"] = pd.to_numeric(df["niveau_vie_median"], errors="coerce")

    print(f"\nValeurs manquantes sur niveau_vie_median : {df['niveau_vie_median'].isna().sum()}")
    return df[["code_dept", "departement_rev", "niveau_vie_median"]]


In [ ]:
# Appel de la fonction 
revenu_dept = charger_niveau_vie("niv2021.xlsx")
display(revenu_dept.head())

### 1.4 Jointure des trois sources

In [ ]:
# Jointures successives 
# inner join SINOE + ruralité : on ne garde que les depts présents dans les deux
# left join + revenu : on conserve tous les depts même si le revenu est manquant


df = (
    sinoe_dept
    .merge(ruralite_dept, on="code_dept", how="inner")
    .merge(revenu_dept[["code_dept", "niveau_vie_median"]], on="code_dept", how="left")
)

print(f"Table finale : {len(df)} départements")

# Terme d'interaction ruralité × niveau de vie 
# Produit des deux variables explicatives.
# Il teste si l'effet de la ruralité sur la valorisation varie selon le revenu.
# Si son coefficient est non significatif alors les deux effets sont indépendants.
df["interaction_rural_revenu"] = df["part_communes_rurales_pct"] * df["niveau_vie_median"]

display(df.head())


## 2. Analyse descriptive

Avant toute modélisation, cette section présente une vue d'ensemble des trois variables principales.

In [ ]:
#  Statistiques descriptives 
stats = df[cols_cles].describe().round(2)
stats.index.name = "Statistique"
display(stats)

**Lecture du tableau :**
Le taux de valorisation moyen est d'environ **49 %** avec une dispersion 
importante (écart-type ~10 pts, min 29 % – max 88 %), signe de fortes 
inégalités inter-départementales.
La part de communes rurales est très élevée en moyenne (84 %), mais la 
distribution est très asymétrique : la médiane est à 93 %, ce qui reflète 
une France majoritairement rurale en nombre de communes, avec quelques 
grands départements urbains qui tirent la moyenne vers le bas (min = 0 pour 
Paris).
Le niveau de vie médian est le plus concentré des trois (std = 1722 €, 
soit ~8 % de la moyenne), ce qui limite son pouvoir discriminant dans la 
régression — les départements sont relativement homogènes sur ce plan.

In [ ]:
print("Top 10 — taux de valorisation le plus élevé")
display(
    df[["code_dept", "departement",
        "taux_valo_total_pct", "taux_valo_matiere_pct", "taux_valo_organique_pct",
        "part_communes_rurales_pct"]]
    .sort_values("taux_valo_total_pct", ascending=False)
    .head(10).round(2).reset_index(drop=True)
)


In [ ]:
print("\nTop 10 (des plus bas) — taux de valorisation le plus bas")
display(
    df[["code_dept", "departement",
        "taux_valo_total_pct", "taux_valo_matiere_pct", "taux_valo_organique_pct",
        "part_communes_rurales_pct"]]
    .sort_values("taux_valo_total_pct", ascending=True)
    .head(10).round(2).reset_index(drop=True)
)

**Analyse :** les départements en tête de classement sont majoritairement ruraux, ce qui va dans le sens de l'hypothèse territoriale. Les derniers du classement sont souvent des départements à forte concentration urbaine.
On peut regarder ce que cela donne en séparant les deux types de valorisation.
 

In [ ]:
#  valorisation MATIÈRE (recyclage) ────────────────────────────────
print("Top 10 — valorisation matière (recyclage)")
display(
    df[["code_dept", "departement", "taux_valo_matiere_pct", "taux_valo_organique_pct"]]
    .sort_values("taux_valo_matiere_pct", ascending=False)
    .head(10).round(2).reset_index(drop=True)
)


In [ ]:
print("Top 10 les plus bas— valorisation matière (recyclage)")
display(
    df[["code_dept", "departement", "taux_valo_matiere_pct", "taux_valo_organique_pct"]]
    .sort_values("taux_valo_matiere_pct", ascending=True)
    .head(10).round(2).reset_index(drop=True)
)


In [ ]:
#  valorisation ORGANIQUE (compostage) ─────────────────────────────
print("Top 10 — valorisation organique (compostage / méthanisation)")
display(
    df[["code_dept", "departement", "taux_valo_matiere_pct", "taux_valo_organique_pct"]]
    .sort_values("taux_valo_organique_pct", ascending=False)
    .head(10).round(2).reset_index(drop=True)
)


In [ ]:
print("Top 10 (les plus bas)— valorisation organique (compostage / méthanisation)")
display(
    df[["code_dept", "departement", "taux_valo_matiere_pct", "taux_valo_organique_pct"]]
    .sort_values("taux_valo_organique_pct", ascending=True)
    .head(10).round(2).reset_index(drop=True)
)

## 3. Visualisations

### 3.1 Distributions des variables

In [ ]:
# Configuration des variables à afficher
# Chaque tuple contient : (nom de colonne, couleur, libellé affiché)
configs_distrib = [
    ("taux_valo_total_pct",       "#2ecc71", "Valorisation totale (%)"),
    ("taux_valo_matiere_pct",     "#27ae60", "Valorisation matière (%)"),
    ("taux_valo_organique_pct",   "#16a085", "Valorisation organique (%)"),
    ("part_communes_rurales_pct", "#e67e22", "Part communes rurales (%)"),
    ("niveau_vie_median",         "#3498db", "Niveau de vie médian (€/an)"),
]

In [ ]:
#On fait des histogrammes  (5)
fig, axes = plt.subplots(5, 1, figsize=(10, 20))

for ax, (col, color, label) in zip(axes, configs_distrib):
    # Histogramme de la distribution
    ax.hist(df[col].dropna(), bins=15, color=color, edgecolor="white", alpha=0.85)

    # Ligne verticale de la médiane
    mediane = df[col].median()
    ax.axvline(mediane, color="black", linestyle="--", linewidth=1.2,
               label=f"Médiane = {mediane:.1f}")

    ax.set_title(label, fontsize=10)
    ax.set_xlabel(label)
    ax.set_ylabel("Nombre de départements")
    ax.legend(fontsize=8)

plt.suptitle("Distribution des variables principales (par département)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Analyse :** La distribution du taux de valorisation est relativement symétrique autour de la médiane. La ruralité présente une distribution bimodale (départements très ruraux vs. très urbains), ce qui pourra générer une relation non-linéaire avec la valorisation. Le niveau de vie médian est quant à lui concentré autour d'une plage relativement étroite, avec quelques départements à fort revenu.

### 3.2 Nuages de points : ruralité, niveau de vie et valorisation

In [ ]:
# Fonction : nuage de points avec régression 
def scatter_regression(ax, x_col, y_col,x_label, y_label, color, n_outliers=3):
    """Nuage de points avec droite de régression et annotation des outliers.
    Les outliers annotés sont les départements dont l'écart à la droite de
    régression (résidu) est le plus grand en valeur absolue."""
    
    # Suppression des lignes avec valeurs manquantes
    tmp = df[[x_col, y_col, "departement"]].dropna()

    # Nuage de points
    ax.scatter(tmp[x_col], tmp[y_col], alpha=0.6, color=color, edgecolors="white", s=50)

    # Calcul et tracé de la droite de régression (degré 1 = linéaire)
    pente, ordonnee = np.polyfit(tmp[x_col], tmp[y_col], 1)
    x_range = np.linspace(tmp[x_col].min(), tmp[x_col].max(), 100)
    ax.plot(x_range, pente * x_range + ordonnee,
            "k--", linewidth=1.5, label="Régression linéaire")


    # On annote les 3 outliers les plus éloignés de la droite
    residus = tmp[y_col] - (pente * tmp[x_col] + ordonnee)
    outliers = residus.abs().nlargest(n_outliers).index
    for idx in outliers:
        ax.annotate(
            tmp.loc[idx, "departement"],
            xy=(tmp.loc[idx, x_col], tmp.loc[idx, y_col]),
            fontsize=7, xytext=(4, 4), textcoords="offset points"
        )

    # Corrélation de Pearson affichée avec le titre
    r, p = pearsonr(tmp[x_col], tmp[y_col])
    ax.set_title(f"{y_label} ~ {x_label}\nr = {r:.3f}  (p = {p:.4f})", fontsize=9)
    ax.legend(fontsize=8)


In [ ]:
# Grille 3×2 : 3 indicateurs × 2 variables explicatives 
# Lignes : valorisation totale / matière / organique
# Colonnes : ruralité / niveau de vie

fig, axes = plt.subplots(3, 2, figsize=(14, 15))

lignes = [
    ("taux_valo_total_pct",     "Valorisation totale (%)"),
    ("taux_valo_matiere_pct",   "Valorisation matière (%)"),
    ("taux_valo_organique_pct", "Valorisation organique (%)"),
]
colonnes = [
    ("part_communes_rurales_pct", "Part de communes rurales (%)", "#e67e22"),
    ("niveau_vie_median",          "Niveau de vie médian (€/an)", "#3498db"),
]

for i, (y_col, y_label) in enumerate(lignes):
    for j, (x_col, x_label, color) in enumerate(colonnes):
        scatter_regression(axes[i][j], x_col, y_col, color)
        axes[i][j].set_xlabel(x_label)
        axes[i][j].set_ylabel(y_label)

plt.suptitle("Relations bivariées : ruralité & niveau de vie vs. valorisation",
             fontsize=13)
plt.tight_layout()
plt.show()

**Analsye :**
- **Ruralité (gauche)** : la droite de régression est  ascendante, confirmant une relation positive entre part de communes rurales et taux de valorisation. Le coefficient de corrélation de Pearson est modéré mais significatif. Cela se retrouve aussi bien pour le total que por les deux types de valorisation.
- **Niveau de vie (droite)** : la pente est quasi-nulle et la p-value dépasse le seuil de 5 %. Il n'y a pas de relation linéaire claire entre le revenu médian et la valorisation des déchets à l'échelle départementale.
- Les départements annotés (outliers) sont ceux dont les résidus sont les plus importants, c'est-à-dire ceux dont le comportement s'écarte le plus de la tendance générale — ils méritent une attention particulière.

### 3.3 Heatmap de corrélation
Petit rappel :
La corrélation de Pearson mesure la "justesse" d'une relation **linéaire** :
 - +1 : relation positive parfaite
 -  0 : aucune relation linéaire
 - −1 : relation négative parfaite



In [ ]:
# Calcul de la matrice de corrélation 
vars_corr  = ["taux_valo_total_pct", "taux_valo_matiere_pct", "taux_valo_organique_pct",
              "part_communes_rurales_pct", "niveau_vie_median"]
labels_corr = ["Valo. totale", "Valo. matière", "Valo. organique",
               "Ruralité", "Niveau de vie"]


# dropna() : on retire les départements avec des valeurs manquantes
# pour que toutes les corrélations soient calculées sur le même ensemble, sinon cela ne fonctionne pas et garde des cases blanches
mat_corr = df[vars_corr].dropna().corr().round(3)

In [ ]:
# Affichage de la heatmap 
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    mat_corr,
    annot=True,         
    fmt=".3f",
    cmap="RdYlGn",     
    vmin=-1, vmax=1,
    xticklabels=labels_corr,
    yticklabels=labels_corr,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 11},  # taille des valeurs dans les cellules
    ax=ax,
)
#  annot permet de mettre les valeurs dans les cases et cmap est définit pour que le rouge soit pour le négatif et le vert pour le positif
ax.set_title("Matrice de corrélation de Pearson", fontsize=13)
ax.tick_params(axis="x", labelsize=11)  
ax.tick_params(axis="y", labelsize=11) 
plt.tight_layout()
plt.show()

**Lecture :** La heatmap confirme que la corrélation la plus forte avec le taux de valorisation est celle de la ruralité (valeur positive modérée). On note également une corrélation négative entre ruralité et niveau de vie — les départements ruraux sont en moyenne moins riches — ce qui soulève un risque de **colinéarité partielle** entre les deux variables explicatives. Ce point sera discuté dans la régression multiple.

### 3.4 Analyse cartographique : les cartes choroplèthes

Rappel : Une carte choroplèthe colorie chaque département selon l'intensité d'une variable : plus la couleur est foncée, plus la valeur est élevée.

In [ ]:
# Chargement du fond de carte 
# Le GeoJSON contient les contours géographiques des 96 départements métropolitains.
carte_base = gpd.read_file(GEOJSON_URL)

# Harmonisation du code pour la jointure
carte_base["code"] = carte_base["code"].astype(str).str.zfill(2)
df["code_dept"]    = df["code_dept"].astype(str).str.zfill(2)

# Jointure : on enrichit chaque polygone géographique avec nos données
carte = carte_base.merge(df, left_on="code", right_on="code_dept", how="left")
print(f"Carte chargée : {len(carte)} départements")

In [ ]:
# Cartes choroplèthes : 5 variables l'une après l'autre
configs_cartes = [
    ("taux_valo_total_pct",       "Greens",  "Valorisation totale (%)"),
    ("taux_valo_matiere_pct",     "YlGn",    "Valorisation matière (%)"),
    ("taux_valo_organique_pct",   "BuGn",    "Valorisation organique (%)"),
    ("part_communes_rurales_pct", "Oranges", "Part communes rurales (%)"),
    ("niveau_vie_median",         "Blues",   "Niveau de vie médian (€)"),
]

fig, axes = plt.subplots(5, 1, figsize=(10, 40))

for ax, (col, cmap, title) in zip(axes, configs_cartes):
    carte.plot(
        column=col,
        cmap=cmap,
        linewidth=0.4,
        edgecolor="white",
        legend=True,
        legend_kwds={"shrink": 0.6, "label": title},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Donnée manquante"},
    )
    ax.set_title(title, fontsize=10)
    ax.axis("off")   

plt.suptitle("Répartition spatiale des variables à l'échelle départementale",
             fontsize=13)
plt.tight_layout()
plt.show()

**Lecture des cartes :**
- La carte de valorisation (verte) révèle une **concentration des bons performeurs dans les zones rurales** (diagonale nord-est / sud-ouest et Massif Central), alors que les grandes métropoles et leurs couronnes affichent des taux plus faibles.
- La carte de ruralité (orange) montre une structure spatiale similaire, ce qui renforce visuellement la corrélation positive entre les deux variables.
- La carte de niveau de vie (bleue) présente un gradient différent, plus concentré autour de l'Île-de-France et de quelques métropoles. Sa géographie est donc **distincte** de celle de la valorisation, cohérent avec l'absence de corrélation significative observée statistiquement.

### 3.5 Carte typologique : 4 profils territoriaux

 On croise deux dimensions binaires :
 - Rural / Urbain (au-dessus / en-dessous de la médiane nationale de ruralité)
 - Valorisation forte / faible (au-dessus / en-dessous de la médiane nationale)

 On identifie 4 profils : les cas "rural / valorisation faible" sont particulièrement intéressants car ils montrent que la ruralité seule ne suffit pas.


In [ ]:
#  Calcul des seuils (médianes nationales) 
med_valo  = df["taux_valo_total_pct"].median()
med_rural = df["part_communes_rurales_pct"].median()

print(f"Seuil valorisation : {med_valo:.1f}%")
print(f"Seuil ruralité     : {med_rural:.1f}%")

In [ ]:
# Attribution d'un profil à chaque département 
def attribuer_profil(row):
    """Retourne le profil territorial d'un département selon ses médianes."""
    valo  = row["taux_valo_total_pct"]        >= med_valo
    rural = row["part_communes_rurales_pct"]  >= med_rural

    if   not rural and not valo: return "Urbain / valorisation faible"
    elif not rural and     valo: return "Urbain / valorisation forte"
    elif     rural and not valo: return "Rural  / valorisation faible"
    else:                        return "Rural  / valorisation forte"

# pd.Categorical avec ordered=True assure un tri cohérent dans les tableaux
ordre_profils = [
    "Urbain / valorisation faible",
    "Urbain / valorisation forte",
    "Rural  / valorisation faible",
    "Rural  / valorisation forte",
]
df["profil"] = pd.Categorical(
    df.apply(attribuer_profil, axis=1),
    categories=ordre_profils,
    ordered=True,
)

In [ ]:
#On complete les profils 
print("Répartition des profils :")
display(df["profil"].value_counts().rename("Nombre de départements").to_frame())


In [ ]:
#  On trace la carte typologique
# Palette : rouge/vert selon la valorisation, clair/foncé selon la ruralité
couleurs_profil = {
    "Urbain / valorisation faible": "#f4a3a3",  # rose clair
    "Urbain / valorisation forte":  "#a8ddb5",  # vert clair
    "Rural  / valorisation faible": "#b30000",  # rouge foncé
    "Rural  / valorisation forte":  "#006d2c",  # vert foncé
}

# Rechargement de la carte avec la colonne profil
carte2 = carte_base.merge(df, left_on="code", right_on="code_dept", how="left")
carte2["couleur"] = carte2["profil"].map(couleurs_profil)

fig, ax = plt.subplots(figsize=(9, 10))

# On passe color= (discret) et non column= (gradient continu)
carte2.plot(
    color=carte2["couleur"].astype(object).fillna("lightgrey"),
    linewidth=0.5,
    edgecolor="white",
    ax=ax,
)

# Légende manuelle avec des patches colorés
patches = [
    mpatches.Patch(facecolor=v, edgecolor="grey", label=k)
    for k, v in couleurs_profil.items()
]
patches.append(mpatches.Patch(facecolor="lightgrey", edgecolor="grey",
                               label="Donnée manquante"))
ax.legend(handles=patches, title="Profil territorial",
          loc="lower left", frameon=True, fontsize=9)
ax.set_title("Profils territoriaux : ruralité x valorisation des déchets",
             fontsize=12)
ax.axis("off")
plt.show()

**Analyse :** A faire

## 4. Modélisation

### 4.1 Corrélations bivariées

In [ ]:
# Fonction : affichage des corrélations 
def afficher_correlations(x_col, y_col, x_label, y_label):
    """
    Calcule et affiche les corrélations de Pearson et Spearman.

    Pearson  : mesure l'association LINÉAIRE. Sensible aux outliers.
    Spearman : mesure l'association MONOTONE (sur les rangs). Plus robuste.

    Si les deux divergent → la relation n'est pas strictement linéaire,
    ou des départements extrêmes "tirent" la corrélation de Pearson.
    """
    tmp = df[[x_col, y_col]].dropna()

    r_p, p_p = pearsonr(tmp[x_col],  tmp[y_col])
    r_s, p_s = spearmanr(tmp[x_col], tmp[y_col])

    sig = "significatif" if p_p < 0.05 else "non significatif"

    print(f"  {x_label} X {y_label}")
    print(f"  Pearson  r = {r_p:.3f}  (p = {p_p:.6f})")
    print(f"  Spearman ρ = {r_s:.3f}  (p = {p_s:.6f})")
    print(f"  → {sig} au seuil 5 %\n")


In [ ]:
# Corrélations : ruralité vs les 3 indicateurs 
print("=" * 60)
print("RURALITÉ X VALORISATION")
print("=" * 60)
afficher_correlations("part_communes_rurales_pct", "taux_valo_total_pct",
                      "Ruralité", "Valorisation totale")
afficher_correlations("part_communes_rurales_pct", "taux_valo_matiere_pct",
                      "Ruralité", "Valorisation matière")
afficher_correlations("part_communes_rurales_pct", "taux_valo_organique_pct",
                      "Ruralité", "Valorisation organique")


In [ ]:
# Corrélations : niveau de vie vs les 3 indicateurs 
print("=" * 60)
print("NIVEAU DE VIE X VALORISATION")
print("=" * 60)
afficher_correlations("niveau_vie_median", "taux_valo_total_pct",
                      "Niveau de vie", "Valorisation totale")
afficher_correlations("niveau_vie_median", "taux_valo_matiere_pct",
                      "Niveau de vie", "Valorisation matière")
afficher_correlations("niveau_vie_median", "taux_valo_organique_pct",
                      "Niveau de vie", "Valorisation organique")

**Interprétation :** La ruralité est significativement corrélée à la valorisation (Pearson et Spearman concordants). Le niveau de vie ne l'est pas. On note également que ruralité et niveau de vie sont négativement corrélés : les territoires ruraux ont tendance à être moins riches. Cette structure de corrélation sera à surveiller dans la régression multiple.

⚠️ **Point d'attention Pour ruralité x Valorisation** : la corrélation de Pearson (r = 0.425, p < 0.001) 
et celle de Spearman (ρ = 0.139, p = 0.177) donnent des conclusions 
opposées. Cela s'explique par la forme de la relation : Pearson capte 
l'association linéaire, Spearman l'association monotone sur les rangs.

La distribution très asymétrique de la ruralité (beaucoup de départements 
à > 90 %, quelques-uns à 0 %) crée des cas extrêmes qui "gonflent" le r 
de Pearson. Le résultat de Spearman invite à la prudence : la relation 
n'est pas robuste sur l'ensemble de la distribution. C'est une raison 
supplémentaire pour interpréter le modèle de régression avec nuance.

### 4.2 Régression Régressions OLS — les trois indicateurs séparément

On estime trois modèles emboîtés pour évaluer l'apport de chaque variable :
- **Modèle 1** : ruralité seule
- **Modèle 2** : ruralité + niveau de vie
- **Modèle 3** : ruralité + niveau de vie + **terme d'interaction** (ruralité × niveau de vie, centré)

Le terme d'interaction teste si l'effet de la ruralité sur la valorisation **varie selon le niveau de richesse du département**.

In [ ]:
# Fonction : construction et affichage de 3 modèles emboîtés 
def regression_ols(var_y, label_y):
    """
    Estime 3 modèles OLS emboîtés et affiche un tableau de synthèse.

    Modèle 1 : ruralité seule
    Modèle 2 : ruralité + niveau de vie
    Modèle 3 : ruralité + niveau de vie + interaction (variables centrées)

    Modèles "emboîtés" : chaque modèle ajoute une variable au précédent.
    On compare le R² ajusté pour évaluer l'apport marginal de chaque variable.
    Si le R² ajusté n'augmente pas, la variable ajoutée n'est pas utile.
    """
    cols = [var_y, "part_communes_rurales_pct", "niveau_vie_median",
            "interaction_rural_revenu"]
    tmp = df[cols].dropna()
    y   = tmp[var_y]

    # sm.add_constant() ajoute une colonne de 1 pour l'ordonnée à l'origine
    X1 = sm.add_constant(tmp[["part_communes_rurales_pct"]])
    X2 = sm.add_constant(tmp[["part_communes_rurales_pct", "niveau_vie_median"]])
    X3 = sm.add_constant(tmp[["part_communes_rurales_pct", "niveau_vie_median",
                               "interaction_rural_revenu"]])

    mod1 = sm.OLS(y, X1).fit()
    mod2 = sm.OLS(y, X2).fit()
    mod3 = sm.OLS(y, X3).fit()

    def fmt(mod, var):
        if var not in mod.params: return "—"
        sig = "*" if mod.pvalues[var] < 0.05 else "ns"
        return f"{mod.params[var]:.4f} ({sig})"

    synthese = pd.DataFrame({
        "Modèle 1 (ruralité)": [
            fmt(mod1, "part_communes_rurales_pct"), "—", "—",
            f"{mod1.rsquared:.3f}", f"{mod1.rsquared_adj:.3f}",
        ],
        "Modèle 2 (+revenu)": [
            fmt(mod2, "part_communes_rurales_pct"),
            fmt(mod2, "niveau_vie_median"), "—",
            f"{mod2.rsquared:.3f}", f"{mod2.rsquared_adj:.3f}",
        ],
        "Modèle 3 (+interaction)": [
            fmt(mod3, "part_communes_rurales_pct"),
            fmt(mod3, "niveau_vie_median"),
            fmt(mod3, "interaction_rural_revenu"),
            f"{mod3.rsquared:.3f}", f"{mod3.rsquared_adj:.3f}",
        ],
    }, index=["Part communes rurales", "Niveau de vie médian",
              "Interaction (centré)", "R²", "R² ajusté"])

    print(f"\n{'='*60}")
    print(f"RÉGRESSION OLS — {label_y}")
    print("="*60)
    display(synthese)
    print("* = significatif à 5 % | ns = non significatif")

    return mod1, mod2, mod3


In [ ]:
# Régression sur la valorisation TOTALE 
mod1_tot, mod2_tot, mod3_tot = regression_ols("taux_valo_total_pct",
                                               "Valorisation totale (%)")

In [ ]:
# Régression sur la valorisation MATIÈRE 
mod1_mat, mod2_mat, mod3_mat = regression_ols("taux_valo_matiere_pct",
                                               "Valorisation matière / recyclage (%)")


In [ ]:
#  Régression sur la valorisation ORGANIQUE 
mod1_org, mod2_org, mod3_org = regression_ols("taux_valo_organique_pct",
                                               "Valorisation organique / compostage (%)")


**Interprétation des modèles : à mettre à jour**

**Modèle 1** : La ruralité seule explique **18 %** de la variance du taux 
de valorisation (R² = 0.180, R² ajusté = 0.172). Le coefficient est positif 
et très significatif (β = 0.191, p < 0.001) : une hausse d'un point de la 
part de communes rurales est associée à +0.19 pt de taux de valorisation.

**Modèle 2** : L'ajout du niveau de vie améliore marginalement le R² 
(passage de 18 % à 18.7 %). Le R² **ajusté** recule légèrement (0.172 → 
0.170), ce qui signifie que le niveau de vie n'apporte pas d'information 
utile au-delà du bruit. La ruralité reste significative (β = 0.214, p < 
0.001) ; le niveau de vie reste non significatif (p = 0.156).

**Modèle 3** : L'interaction n'est pas significative (p = 0.233). L'effet 
de la ruralité sur la valorisation ne varie donc pas selon la richesse du 
département. Le R² ajusté remonte légèrement à 0.174 mais reste modéré.
Le modèle 1 reste donc le plus parcimonieux et suffisant pour conclure.

### 4.3 Vérification des hypothèses du modèle OLS

In [ ]:
# Fonction : graphiques de diagnostic 
def diagnostics_ols(modele, titre):
    """
    Vérifie les deux hypothèses principales de la régression OLS :

    1. Homoscédasticité (variance constante des résidus)
       → Graphique résidus vs valeurs ajustées : le nuage doit être aléatoire
         autour de 0, sans forme en entonnoir.

    2. Normalité des résidus
       → Q-Q plot : les points doivent suivre la diagonale.
       → Test de Shapiro-Wilk : H0 = normalité. Si p < 0.05 → non normal.
         Dans ce cas, les p-values de la régression sont à interpréter avec prudence.
    """
    residus = modele.resid          # différence entre valeur observée et prédite
    fitted  = modele.fittedvalues   # valeurs prédites par le modèle

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Graphique 1 : résidus vs valeurs ajustées
    axes[0].scatter(fitted, residus, alpha=0.6, color="#7f8c8d",
                    edgecolors="white", s=40)
    axes[0].axhline(0, color="red", linestyle="--", linewidth=1)
    axes[0].set_xlabel("Valeurs ajustées (prédites par le modèle)")
    axes[0].set_ylabel("Résidus (observé - prédit)")
    axes[0].set_title("Résidus vs valeurs ajustées")

    # Graphique 2 : Q-Q plot
    # line="s" : droite de référence basée sur l'écart-type des résidus
    sm.qqplot(residus, line="s", ax=axes[1], alpha=0.6)
    axes[1].set_title("Q-Q plot (normalité des résidus)")

    # Graphique 3 : histogramme des résidus
    axes[2].hist(residus, bins=15, color="#95a5a6", edgecolor="white")
    axes[2].set_xlabel("Résidus")
    axes[2].set_ylabel("Fréquence")
    axes[2].set_title("Distribution des résidus")

    plt.suptitle(f"Diagnostics — {titre}", fontsize=12)
    plt.tight_layout()
    plt.show()

    # Test de Shapiro-Wilk
    stat, p_sw = shapiro(residus)
    print(f"Shapiro-Wilk : W = {stat:.4f}, p = {p_sw:.4f}")
    if p_sw > 0.05:
        print("→ Résidus normaux (p > 0.05)")
    else:
        print("→ Résidus non normaux (p < 0.05) attention — interpréter les IC avec prudence")


In [ ]:
# Diagnostics — valorisation totale
diagnostics_ols(mod3_tot, "Valorisation totale")


In [ ]:
#  Diagnostics — valorisation matière 
diagnostics_ols(mod3_mat, "Valorisation matière")


In [ ]:
#  Diagnostics — valorisation organique 
diagnostics_ols(mod3_org, "Valorisation organique")


**Lecture des diagnostics : A mettre à jour**
- *Résidus vs valeurs ajustées* : le nuage est globalement centré sur 0 
  mais quelques points extrêmes en haut à gauche (forts résidus positifs 
  pour les hauts taux de valorisation) signalent une légère 
  hétéroscédasticité.
- *Q-Q plot* : les queues s'écartent de la diagonale, surtout à droite — 
  quelques départements outliers (Vendée, Saône-et-Loire) tirent la 
  distribution vers le haut.
- *Shapiro-Wilk* : W = 0.9225, p < 0.001 → on rejette la normalité. Les 
  p-values et intervalles de confiance de la régression sont donc à 
  interpréter avec prudence. Une solution serait de tester une 
  transformation log du taux de valorisation ou d'utiliser des erreurs 
  standard robustes (HC3).

## 5. Conclusion générale

Cette étude avait pour objectif d'identifier les facteurs territoriaux et socio-économiques associés à la valorisation des déchets ménagers à l'échelle départementale en France.

**Principaux résultats :**

1. **La ruralité est positivement et significativement associée au taux de valorisation.** Cette relation se confirme à travers l'analyse descriptive, les corrélations, les cartes choroplèthes et la régression. Elle peut s'expliquer par une plus grande pratique du compostage et de la valorisation organique en milieu rural, ainsi qu'une moindre densité de déchets à gérer.

2. **Le niveau de vie médian n'est pas un déterminant significatif.** L'absence de relation entre revenu et valorisation suggère que les performances en matière de recyclage ne sont pas davantage le fait des territoires riches, mais qu'elles dépendent davantage de l'organisation territoriale.

3. **Le terme d'interaction n'est pas significatif** (p = 0.233) : l'effet de la ruralité sur la valorisation est indépendant du niveau de richesse du département. Les territoires ruraux pauvres et les territoires ruraux aisés valorisent leurs déchets dans des proportions similaires.



**Limites :**
- L'indicateur de ruralité n'est pas pondéré par la population, ce qui sur-représente les communes faiblement peuplées.
- Le modèle explique une part limitée de la variance (R² modéré) : d'autres facteurs (densité d'équipements, politiques locales, part de maisons individuelles avec jardin) mériteraient d'être intégrés.
- L'analyse est transversale (une seule année) : une approche longitudinale permettrait de contrôler des effets fixes départementaux.

**Pistes d'approfondissement :** intégration d'indicateurs de mobilité verte, de densité de déchèteries, ou de la part de logements individuels, evolution dans le temps.